<a href="https://colab.research.google.com/github/MurphyKlein/CS4782_final_project/blob/main/notebook/tran.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

BASE_DIR = "/content/drive/MyDrive/DL_Final_Project"

# ─────────────────────────────────────────────
# 1. WEATHER DATASET (Merged CSVs)
# ─────────────────────────────────────────────
WEATHER_DIR = f"{BASE_DIR}/weather_data"

def load_all_weather(directory):
    all_files = [f for f in os.listdir(directory) if f.endswith('.csv')]
    print(f"[Weather] Found {len(all_files)} files: {all_files}")

    dfs = []
    for file in all_files:
        filepath = os.path.join(directory, file)
        df = pd.read_csv(filepath, encoding='unicode_escape')
        print(f"  -> Loaded {file}: {df.shape}")

        # drop any datetime / string columns
        date_cols = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower()]
        if date_cols:
            df = df.drop(columns=date_cols)

        df = df.select_dtypes(include=[np.number])
        dfs.append(df)

    combined_df = pd.concat(dfs, axis=0, ignore_index=True)
    combined_df = combined_df.dropna()
    print(f"[Weather] Final Merged shape: {combined_df.shape}")
    return combined_df

weather_df = load_all_weather(WEATHER_DIR)

# ─────────────────────────────────────────────
# 2. ELECTRICITY DATASET (LD2011_2014.txt)
# ─────────────────────────────────────────────
ELECTRICITY_DIR = f"{BASE_DIR}/electricity_data"
ELECTRICITY_FILE = os.path.join(ELECTRICITY_DIR, "LD2011_2014.txt")

def load_electricity(filepath):
    print("\n[Electricity] Loading... (this may take a moment)")
    df = pd.read_csv(filepath, sep=';', decimal=',', index_col=0, parse_dates=True)
    print(f"[Electricity] Raw shape: {df.shape}")

    # resample 15-min → hourly
    df = df.resample('1H').mean()

    # Drop columns that contain any 0s (for Electricity only)
    cols_with_zeros = df.columns[(df == 0).any()]
    print(f"[Electricity] Dropping {len(cols_with_zeros)} columns containing zeros.")
    df = df.drop(columns=cols_with_zeros)

    df = df.dropna()
    print(f"[Electricity] Final shape: {df.shape}")
    return df

electricity_df = load_electricity(ELECTRICITY_FILE)

# ─────────────────────────────────────────────
# 3. SPLIT + SCALE
# ─────────────────────────────────────────────
def split_and_scale(df, train_ratio=0.7, val_ratio=0.1):
    n = len(df)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    train = df.iloc[:n_train].values.astype(np.float32)
    val = df.iloc[n_train : n_train + n_val].values.astype(np.float32)
    test = df.iloc[n_train + n_val:].values.astype(np.float32)

    scaler = StandardScaler().fit(train)
    return scaler.transform(train), scaler.transform(val), scaler.transform(test), scaler

print("\n── Weather splits ──")
w_train, w_val, w_test, w_scaler = split_and_scale(weather_df)
print(f"  train {w_train.shape}  val {w_val.shape}  test {w_test.shape}")

print("\n── Electricity splits ──")
e_train, e_val, e_test, e_scaler = split_and_scale(electricity_df)
print(f"  train {e_train.shape}  val {e_val.shape}  test {e_test.shape}")

print("\nAll done — data ready for DataLoader.")

[Weather] Found 11 files: ['mpi_roof_2020a.csv', 'mpi_roof_2020b.csv', 'mpi_roof_2021a.csv', 'mpi_roof_2021b.csv', 'mpi_roof_2022a.csv', 'mpi_roof_2022b.csv', 'mpi_roof_2023a.csv', 'mpi_roof_2023b.csv', 'mpi_roof_2024.csv', 'mpi_roof_2025.csv', 'mpi_roof.csv']
  -> Loaded mpi_roof_2020a.csv: (26200, 22)
  -> Loaded mpi_roof_2020b.csv: (26496, 22)
  -> Loaded mpi_roof_2021a.csv: (26064, 22)
  -> Loaded mpi_roof_2021b.csv: (26496, 22)
  -> Loaded mpi_roof_2022a.csv: (26070, 22)
  -> Loaded mpi_roof_2022b.csv: (26415, 22)
  -> Loaded mpi_roof_2023a.csv: (26061, 22)
  -> Loaded mpi_roof_2023b.csv: (26639, 22)
  -> Loaded mpi_roof_2024.csv: (52704, 22)
  -> Loaded mpi_roof_2025.csv: (52560, 22)
  -> Loaded mpi_roof.csv: (17403, 22)
[Weather] Final Merged shape: (333108, 21)

[Electricity] Loading... (this may take a moment)
[Electricity] Raw shape: (140256, 370)
[Electricity] Dropping 280 columns containing zeros.
[Electricity] Final shape: (35065, 90)

── Weather splits ──


/tmp/ipykernel_2096/784139141.py:50: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df = df.resample('1H').mean()


  train (233175, 21)  val (33310, 21)  test (66623, 21)

── Electricity splits ──
  train (24545, 90)  val (3506, 90)  test (7014, 90)

All done — data ready for DataLoader.


In [ ]:
class Config:
    # ── paths ──────────────────────────────────────────────────────────────
    project_path   = '/content/drive/MyDrive/DL_Final_Project'
    data_dir       = '/content/drive/MyDrive/DL_Final_Project/weather_data'
    checkpoint_dir = '/content/drive/MyDrive/DL_Final_Project/lin_prob_checkpoints'

    # ── sequence ───────────────────────────────────────────────────────────
    seq_len   = 512
    pred_lens = [96, 192, 336, 720]

    # ── patching (non-overlapping, as in SSL section of paper) ─────────────
    patch_len = 12
    stride    = 12     # non-overlapping  →  N ≈ 42 patches

    # ── transformer ────────────────────────────────────────────────────────
    d_model  = 128
    n_heads  = 16
    n_layers = 3
    d_ff     = 256
    dropout  = 0.2

    # ── pre-training ───────────────────────────────────────────────────────
    mask_ratio      = 0.40
    pretrain_epochs = 100
    pretrain_lr     = 1e-4
    pretrain_bs     = 128

    # ── linear probing ─────────────────────────────────────────────────────
    lp_epochs = 20
    lp_lr     = 1e-4
    lp_bs     = 128

    # ── early stopping ─────────────────────────────────────────────────────
    patience = 10

    # ── data split ─────────────────────────────────────────────────────────
    train_ratio = 0.7
    val_ratio   = 0.1

cfg = Config()
os.makedirs(cfg.checkpoint_dir, exist_ok=True)
print("Config ready. Checkpoints →", cfg.checkpoint_dir)

Config ready. Checkpoints → /content/drive/MyDrive/DL_Final_Project/lin_prob_checkpoints


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class PretrainDataset(Dataset):
    """Returns [M, L] windows for masked autoencoder pre-training."""
    def __init__(self, data, seq_len):
        self.data    = torch.FloatTensor(data)   # [T, M]
        self.seq_len = seq_len

    def __len__(self):
        return max(0, len(self.data) - self.seq_len + 1)

    def __getitem__(self, idx):
        return self.data[idx : idx + self.seq_len].T   # [M, L]

class ForecastDataset(Dataset):
    """Returns ([M, L], [M, pred_len]) pairs."""
    def __init__(self, data, seq_len, pred_len):
        self.data     = torch.FloatTensor(data)
        self.seq_len  = seq_len
        self.pred_len = pred_len

    def __len__(self):
        return max(0, len(self.data) - self.seq_len - self.pred_len + 1)

    def __getitem__(self, idx):
        x = self.data[idx           : idx + self.seq_len]
        y = self.data[idx+self.seq_len : idx+self.seq_len+self.pred_len]
        return x.T, y.T   # [M, L], [M, pred_len]

def pretrain_loaders(train_data, val_data):
    tr = DataLoader(PretrainDataset(train_data, cfg.seq_len),
                    batch_size=cfg.pretrain_bs, shuffle=True,  drop_last=True)
    va = DataLoader(PretrainDataset(val_data,   cfg.seq_len),
                    batch_size=cfg.pretrain_bs, shuffle=False, drop_last=False)
    return tr, va

def forecast_loaders(train_data, val_data, test_data, pred_len):
    kw = dict(drop_last=False)
    tr = DataLoader(ForecastDataset(train_data, cfg.seq_len, pred_len),
                    batch_size=cfg.lp_bs, shuffle=True,  drop_last=True)
    va = DataLoader(ForecastDataset(val_data,   cfg.seq_len, pred_len),
                    batch_size=cfg.lp_bs, shuffle=False, **kw)
    te = DataLoader(ForecastDataset(test_data,  cfg.seq_len, pred_len),
                    batch_size=cfg.lp_bs, shuffle=False, **kw)
    return tr, va, te

print("Dataset classes and updated loader functions ready.")

Dataset classes and updated loader functions ready.


In [ ]:
import torch
import torch.nn as nn

# ── helpers ────────────────────────────────────────────────────────────────
def instance_norm(x, eps=1e-5):
    """x: [B, M, L] -> (x_norm, mean [B,M,1], std [B,M,1])"""
    mu  = x.mean(dim=-1, keepdim=True)
    std = x.std(dim=-1,  keepdim=True) + eps
    return (x - mu) / std, mu, std

def instance_denorm(x, mu, std):
    return x * std + mu


# ── Backbone ───────────────────────────────────────────────────────────────
class PatchTSTBackbone(nn.Module):
    """Channel-independent: patch projection + positional emb + Transformer."""

    def __init__(self, patch_len, stride, seq_len,
                 d_model, n_heads, n_layers, d_ff, dropout):
        super().__init__()
        self.patch_len   = patch_len
        self.stride      = stride
        # num_patches formula from the paper: floor((L-P)/S) + 2
        self.num_patches = (seq_len - patch_len) // stride + 2
        self.d_model     = d_model

        self.proj    = nn.Linear(patch_len, d_model)
        self.pos_emb = nn.Parameter(
            torch.zeros(1, self.num_patches, d_model))
        nn.init.trunc_normal_(self.pos_emb, std=0.02)

        self.drop = nn.Dropout(dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_ff, dropout=dropout,
            batch_first=True, norm_first=False)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

    def patchify(self, x):
        """x: [N, L] -> patches [N, num_patches, patch_len]"""
        pad  = x[:, -1:].expand(-1, self.stride)
        x_p  = torch.cat([x, pad], dim=1)
        return x_p.unfold(1, self.patch_len, self.stride)   # [N, NP, P]

    def forward(self, x):
        """x: [B*M, L] -> [B*M, num_patches, d_model]"""
        z = self.proj(self.patchify(x)) + self.pos_emb
        z = self.drop(z)
        return self.encoder(z)


# ── Pre-training (Masked Autoencoder) ─────────────────────────────────────
class PatchTSTMaskedAE(nn.Module):
    def __init__(self, backbone, mask_ratio):
        super().__init__()
        self.backbone   = backbone
        self.mask_ratio = mask_ratio
        self.head       = nn.Linear(backbone.d_model, backbone.patch_len)

    def forward(self, x):
        """x: [B, M, L]  ->  scalar MSE loss on masked patches"""
        B, M, L = x.shape
        x_norm, _, _ = instance_norm(x)
        bm = B * M
        xf = x_norm.reshape(bm, L)

        patches = self.backbone.patchify(xf)     # [B*M, N, P]
        N, P    = patches.shape[1], patches.shape[2]

        n_mask = int(N * self.mask_ratio)
        noise  = torch.rand(bm, N, device=x.device)
        ids    = torch.argsort(noise, dim=1)
        bool_mask = torch.zeros(bm, N, dtype=torch.bool, device=x.device)
        bool_mask.scatter_(1, ids[:, :n_mask], True)

        p_in = patches.clone()
        p_in[bool_mask] = 0.0

        z    = self.backbone.proj(p_in) + self.backbone.pos_emb
        z    = self.backbone.drop(z)
        z    = self.backbone.encoder(z)
        pred = self.head(z)

        loss = ((pred - patches) ** 2)[bool_mask].mean()
        return loss


# ── Forecasting head (linear probe / fine-tune) ───────────────────────────
class PatchTSTForecast(nn.Module):
    def __init__(self, backbone, pred_len, freeze_backbone=True):
        super().__init__()
        self.backbone = backbone
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad_(False)
        flat_dim = backbone.num_patches * backbone.d_model
        self.head = nn.Linear(flat_dim, pred_len)

    def forward(self, x, return_stats=False):
        """x: [B, M, L] -> [B, M, pred_len]  (in instance-normalized space)"""
        B, M, L = x.shape
        x_norm, mu, std = instance_norm(x)
        z    = self.backbone(x_norm.reshape(B*M, L))   # [B*M, N, d]
        pred = self.head(z.flatten(1))                 # [B*M, pred_len]
        pred = pred.reshape(B, M, -1)                  # [B, M, pred_len]
        if return_stats:
            return pred, mu, std   # caller handles denorm
        return pred

print(f"Model classes ready.  backbone num_patches = "
      f"{(cfg.seq_len - cfg.patch_len) // cfg.stride + 2}")

Model classes ready.  backbone num_patches = 43


In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, delta=1e-6, path='checkpoint.pt'):
        self.patience = patience
        self.delta    = delta
        self.path     = path
        self.best     = np.inf
        self.counter  = 0
        self.stop     = False

    def __call__(self, val_loss, model):
        if val_loss < self.best - self.delta:
            self.best    = val_loss
            self.counter = 0
            torch.save(model.state_dict(), self.path)
            return True          # saved
        self.counter += 1
        if self.counter >= self.patience:
            self.stop = True
        return False

In [ ]:
import torch.optim as optim
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 65)
print("PHASE 1  –  PRE-TRAINING ON ELECTRICITY")
print("=" * 65)

backbone = PatchTSTBackbone(
    patch_len = cfg.patch_len,
    stride    = cfg.stride,
    seq_len   = cfg.seq_len,
    d_model   = cfg.d_model,
    n_heads   = cfg.n_heads,
    n_layers  = cfg.n_layers,
    d_ff      = cfg.d_ff,
    dropout   = cfg.dropout,
).to(device)

mae_model = PatchTSTMaskedAE(backbone, cfg.mask_ratio).to(device)

pt_ckpt = os.path.join(cfg.checkpoint_dir, 'pretrained_backbone_electricity.pt')
es_pt   = EarlyStopping(patience=cfg.patience, path=pt_ckpt)

opt_pt  = optim.Adam(mae_model.parameters(), lr=cfg.pretrain_lr)
sch_pt  = optim.lr_scheduler.CosineAnnealingLR(opt_pt, T_max=cfg.pretrain_epochs)

# Pre-train on Electricity data
tr_loader, va_loader = pretrain_loaders(e_train, e_val)

pt_train_losses, pt_val_losses = [], []
start = __import__('time').time()

for epoch in range(1, cfg.pretrain_epochs + 1):
    mae_model.train()
    tr_loss = 0.0
    for xb in tr_loader:
        xb = xb.to(device)
        opt_pt.zero_grad()
        loss = mae_model(xb)
        loss.backward()
        nn.utils.clip_grad_norm_(mae_model.parameters(), 1.0)
        opt_pt.step()
        tr_loss += loss.item()
    tr_loss /= len(tr_loader)

    mae_model.eval()
    va_loss = 0.0
    with torch.no_grad():
        for xb in va_loader:
            va_loss += mae_model(xb.to(device)).item()
    va_loss /= len(va_loader)

    sch_pt.step()
    pt_train_losses.append(tr_loss)
    pt_val_losses.append(va_loss)

    saved = es_pt(va_loss, backbone)
    flag  = " ✓" if saved else ""

    if epoch % 10 == 0 or epoch == 1:
        elapsed = __import__('time').time() - start
        print(f"Epoch {epoch:3d}/{cfg.pretrain_epochs}  train={tr_loss:.6f}  val={va_loss:.6f}  [{elapsed:.0f}s]{flag}")

    if es_pt.stop:
        print(f"\nEarly stopping at epoch {epoch} (best val = {es_pt.best:.6f})")
        break

backbone.load_state_dict(torch.load(pt_ckpt, map_location=device))
print("\nLoaded best pre-trained backbone (Electricity).")

PHASE 1  –  PRE-TRAINING ON ELECTRICITY
Epoch   1/100  train=0.911942  val=0.841791  [53s] ✓
Epoch  10/100  train=0.122605  val=0.139622  [507s] ✓
Epoch  20/100  train=0.114258  val=0.131331  [1010s] ✓
Epoch  30/100  train=0.111446  val=0.129385  [1514s] ✓
Epoch  40/100  train=0.109593  val=0.128597  [2018s]
Epoch  50/100  train=0.108158  val=0.126982  [2522s] ✓
Epoch  60/100  train=0.097473  val=0.118850  [3026s] ✓
Epoch  70/100  train=0.093051  val=0.106466  [3530s] ✓
Epoch  80/100  train=0.090953  val=0.101185  [4034s] ✓
Epoch  90/100  train=0.090089  val=0.099592  [4538s]
Epoch 100/100  train=0.090002  val=0.099077  [5042s] ✓

Loaded best pre-trained backbone (Electricity).


In [ ]:
import copy
import torch.optim as optim
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

backbone = PatchTSTBackbone(
    patch_len = cfg.patch_len,
    stride    = cfg.stride,
    seq_len   = cfg.seq_len,
    d_model   = cfg.d_model,
    n_heads   = cfg.n_heads,
    n_layers  = cfg.n_layers,
    d_ff      = cfg.d_ff,
    dropout   = cfg.dropout,
).to(device)

mae_model = PatchTSTMaskedAE(backbone, cfg.mask_ratio).to(device)

pt_ckpt = os.path.join(cfg.checkpoint_dir, 'pretrained_backbone_electricity.pt')
es_pt   = EarlyStopping(patience=cfg.patience, path=pt_ckpt)

opt_pt  = optim.Adam(mae_model.parameters(), lr=cfg.pretrain_lr)
sch_pt  = optim.lr_scheduler.CosineAnnealingLR(opt_pt, T_max=cfg.pretrain_epochs)
backbone.load_state_dict(torch.load(pt_ckpt, map_location=device))

paper_lp = {
    96:  {'MSE': 0.163, 'MAE': 0.216},
    192: {'MSE': 0.205, 'MAE': 0.252},
    336: {'MSE': 0.253, 'MAE': 0.289},
    720: {'MSE': 0.320, 'MAE': 0.336},
}

our_results = {}

print("=" * 65)
print("PHASE 2  –  LINEAR PROBING ON WEATHER")
print("=" * 65)

for pred_len in cfg.pred_lens:
    print(f"\npred_len = {pred_len}")

    bb = copy.deepcopy(backbone)
    model = PatchTSTForecast(bb, pred_len=pred_len, freeze_backbone=True).to(device)

    lp_ckpt = os.path.join(cfg.checkpoint_dir, f'linprob_weather_T{pred_len}.pt')
    es      = EarlyStopping(patience=cfg.patience, path=lp_ckpt)

    trainable = [p for p in model.parameters() if p.requires_grad]
    opt = optim.Adam(trainable, lr=cfg.lp_lr)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.lp_epochs)
    criterion = nn.MSELoss()

    # Forecast/Probe on Weather data
    tr_loader, va_loader, te_loader = forecast_loaders(w_train, w_val, w_test, pred_len)

    for epoch in range(1, cfg.lp_epochs + 1):
        model.train()
        tr_loss = 0.0
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            pred_norm, mu, std = model(xb, return_stats=True)
            pred = instance_denorm(pred_norm, mu, std)
            loss = criterion(pred, yb)
            loss.backward()
            opt.step()
            tr_loss += loss.item()

        model.eval()
        va_loss = 0.0
        with torch.no_grad():
            for xb, yb in va_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred_norm, mu, std = model(xb, return_stats=True)
                pred = instance_denorm(pred_norm, mu, std)
                va_loss += criterion(pred, yb).item()

        sch.step()
        if es(va_loss / len(va_loader), model): pass
        if es.stop: break

    model.load_state_dict(torch.load(lp_ckpt, map_location=device))
    model.eval()
    all_preds, all_trues = [], []
    with torch.no_grad():
        for xb, yb in te_loader:
            pred_norm, mu, std = model(xb.to(device), return_stats=True)
            all_preds.append(instance_denorm(pred_norm, mu, std).cpu().numpy())
            all_trues.append(yb.numpy())

    preds, trues = np.concatenate(all_preds), np.concatenate(all_trues)
    mse, mae = np.mean((preds-trues)**2), np.mean(np.abs(preds-trues))
    our_results[pred_len] = {'MSE': mse, 'MAE': mae}

    p = paper_lp[pred_len]
    print(f"  TEST {pred_len} | MSE: {mse:.4f} (Paper: {p['MSE']:.3f}) | MAE: {mae:.4f} (Paper: {p['MAE']:.3f})")

PHASE 2  –  LINEAR PROBING ON WEATHER

pred_len = 96
  TEST 96 | MSE: 0.2850 (Paper: 0.163) | MAE: 0.2895 (Paper: 0.216)

pred_len = 192
  TEST 192 | MSE: 0.3320 (Paper: 0.205) | MAE: 0.3292 (Paper: 0.252)

pred_len = 336
  TEST 336 | MSE: 0.3843 (Paper: 0.253) | MAE: 0.3692 (Paper: 0.289)

pred_len = 720
  TEST 720 | MSE: 0.4569 (Paper: 0.320) | MAE: 0.4178 (Paper: 0.336)
